<center><b><font size='10'><font color='78aa58'>G</font><font color='6589bf'>V</font>HMR</font></b></center>

<center><b><font size='5'> World-Grounded Human Motion Recovery via <font color='78aa58'>Gravity</font>-<font color='6589bf'>View</font> <font color='c68b5b'>Coordinates</font></font></b></center>

![](https://zju3dv.github.io/gvhmr/img/teaser_v1.png)


<center><b>
<a href=https://zju3dv.github.io/gvhmr>Project Page</a>
|
<a href=https://arxiv.org/abs/2409.06662>Paper</a>
</b></center>

> World-Grounded Human Motion Recovery via Gravity-View Coordinates  
> [Zehong Shen](https://zehongs.github.io/)<sup>\*</sup>,
[Huaijin Pi](https://phj128.github.io/)<sup>\*</sup>,
[Yan Xia](https://isshikihugh.github.io/scholar),
[Zhi Cen](https://scholar.google.com/citations?user=Xyy-uFMAAAAJ),
[Sida Peng](https://pengsida.net/)<sup>†</sup>,
[Zechen Hu](https://zju3dv.github.io/gvhmr),
[Hujun Bao](http://www.cad.zju.edu.cn/home/bao/),
[Ruizhen Hu](https://csse.szu.edu.cn/staff/ruizhenhu/),
[Xiaowei Zhou](https://xzhou.me/)  
> SIGGRAPH Asia 2024


## Installation

Check [INSTALL.md](https://github.com/IsshikiHugh/GVHMR/blob/main/docs/INSTALL.md) if you want to install GVHMR on your own machine.

> Tips: you can fold the section and run the whole installation section at once.

In [1]:
# Controlling notebook is connected to NVIDIA drivers with CUDA. If this doesn't load check that GPU is selected as hardware accelerator under Edit -> Notebook settings.
# Google may shut down the GPU if usage has surpassed the allocation

!nvidia-smi

Thu Oct 15 04:04:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Environment Prepration ~ 15 min

Check [INSTALL.md#Environment](https://github.com/IsshikiHugh/GVHMR/blob/main/docs/INSTALL.md#environment) for further information.

In [2]:
import os
from pathlib import Path

# Clone the repo.
!git clone https://github.com/zju3dv/GVHMR
proj_root = str(Path('GVHMR').absolute())

# Install GVHMR
# FIX: Colab runs Python 3.13, but GVHMR needs Python 3.10.
# GVHMR is installed into a seperate Python 3.10 environment (/content/py310)
%cd {proj_root}
py310 = '/content/py310/bin/python'
!pip install -q uv
!uv venv --seed --python 3.10 /content/py310

# chumpy's setup.py imports pip
!grep -v chumpy requirements.txt > /tmp/requirements_py310.txt

#setuptools<81: newer setuptools removed pkg_resources, which lightning 2.3 needs
!uv pip install --python {py310} --index-strategy unsafe-best-match -r /tmp/requirements_py310.txt "setuptools<81"

#pip installed separetely (without build isolation)
!uv pip install --python {py310} --no-build-isolation chumpy
!uv pip install --python {py310} -e .

Cloning into 'GVHMR'...
remote: Enumerating objects: 378, done.
remote: Counting objects: 100% (209/209), done.
remote: Compressing objects: 100% (156/156), done.
remote: Total 378 (delta 71), reused 53 (delta 53), pack-reused 169 (from 1)
Receiving objects: 100% (378/378), 9.31 MiB | 29.14 MiB/s, done.
Resolving deltas: 100% (89/89), done.
/content/GVHMR
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 80.5 MB/s eta 0:00:00
Using CPython 3.10.22
Creating virtual environment with seed packages at: /content/py310
 + packaging==26.3
 + pip==26.2.1
 + setuptools==84.0.0
 + wheel==0.48.0
Activate with: source /content/py310/bin/activate
Using Python 3.10.22 environment at: /content/py310
Resolved 213 packages in 5.12s
Prepared 212 packages in 1m 10s
Uninstalled 2 packages in 93ms
Installed 213 packages in 1.12s
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiosignal==1.4.0
 + annotated-doc==0.0.5
 + antlr4-python3-runtime==4.9.3
 + anyio==4.15.1
 + argon2-cffi==25.1.0
 + argon2-

In [3]:
# Install DPVO. (Optional)
#%cd third-party/DPVO

#!wget https://gitlab.com/libeigen/eigen/-/archive/3.4.0/eigen-3.4.0.zip
#!unzip -o eigen-3.4.0.zip -d thirdparty && rm -rf eigen-3.4.0.zip

#%pip install torch-scatter -f "https://data.pyg.org/whl/torch-2.3.0+cu121.html"
#%pip install numba pypose

#if 'cuda_home' not in locals():
#  cuda_home = '/usr/local/cuda-12'
#  if not Path(cuda_home).exists():
#    raise FileNotFoundError('CUDA_HOME for cuda 12.x not found!')

# os.environ['CUDA_HOME'] = cuda_home
#  os.environ['PATH'] = os.environ['PATH'] + f':{cuda_home}/bin'

# %pip install -e .
# %cd {proj_root}

### Data Prepration ~ 1 min

Check [INSTALL.md#Inputs&Outputs](https://github.com/IsshikiHugh/GVHMR/blob/main/docs/INSTALL.md#inputs--outputs) for further information.

In [5]:
%cd {proj_root}
!mkdir inputs
!mkdir outputs

!apt install -y -qq aria2
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SMPLer-X/resolve/main/SMPL_NEUTRAL.pkl -d inputs/checkpoints/body_models/smpl -o SMPL_NEUTRAL.pkl
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SMPLer-X/resolve/main/SMPLX_NEUTRAL.npz -d inputs/checkpoints/body_models/smplx -o SMPLX_NEUTRAL.npz
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/GVHMR/resolve/main/dpvo/dpvo.pth -d inputs/checkpoints/dpvo -o dpvo.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/GVHMR/resolve/main/gvhmr/gvhmr_siga24_release.ckpt -d inputs/checkpoints/gvhmr -o gvhmr_siga24_release.ckpt
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/GVHMR/resolve/main/hmr2/epoch%3D10-step%3D25000.ckpt -d inputs/checkpoints/hmr2 -o epoch=10-step=25000.ckpt
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/GVHMR/resolve/main/vitpose/vitpose-h-multi-coco.pth -d inputs/checkpoints/vitpose -o vitpose-h-multi-coco.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/GVHMR/resolve/main/yolo/yolov8x.pt -d inputs/checkpoints/yolo -o yolov8x.pt

/content/GVHMR
mkdir: cannot create directory ‘inputs’: File exists
mkdir: cannot create directory ‘outputs’: File exists
The following additional packages will be installed:
  libaria2-0 libcares2
The following NEW packages will be installed:
  aria2 libaria2-0 libcares2
0 upgraded, 3 newly installed, 0 to remove and 33 not upgraded.
Need to get 1,566 kB of archives.
After this operation, 5,692 kB of additional disk space will be used.
Selecting previously unselected package libcares2:amd64.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../libcares2_1.27.0-1.0ubuntu1_amd64.deb ...
Unpacking libcares2:amd64 (1.27.0-1.0ubuntu1) ...
Selecting previously unselected package libaria2-0:amd64.
Preparing to unpack .../libaria2-0_1.37.0+debian-1build3_amd64.deb ...
Unpacking libaria2-0:amd64 (1.37.0+debian-1build3) ...
Selecting previously unselected package aria2.
Preparing to unpack .../aria2_1.37.0+debian-1build3_amd64.deb ...
Unpacking aria2 

## Run Demo

Use `-s` to skip visual odometry <b><u>if you know the camera is static</u></b>, otherwise the camera will be estimated by DPVO.

We also provide a script demo_folder.py to inference a entire folder.


```shell
python tools/demo/demo.py --video=docs/example_video/tennis.mp4 -s
python tools/demo/demo_folder.py -f inputs/demo/folder_in -d outputs/demo/folder_out -s

```

In [7]:
import io
import base64
from IPython.display import HTML
import cv2 # FIX: Hmr4d is installed in the Python 3.10 env

def display_video(fn):
    cap = cv2.VideoCapture(fn)
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    scale = min(W, 1080) / W
    W, H = int(W * scale), int(H * scale)
    video_encoded = base64.b64encode(io.open(fn, 'rb').read())
    return HTML(data='''<video width="{0}" height="{1}" alt="test" controls>
                        <source src="data:video/mp4;base64,{2}" type="video/mp4" />
                        </video>'''.format(W, H, video_encoded.decode('ascii')))

### Demo Tennis

In [8]:
# Run demo.
video_fn = f'{proj_root}/docs/example_video/tennis.mp4'

# FIX: run demo in Python 3.10 environment
!PYTHONPATH= MPLBACKEND=Agg {py310} {proj_root}/tools/demo/demo.py --video={video_fn} -s

[10/15 04:16:34][INFO] [Input]: /content/GVHMR/docs/example_video/tennis.mp4
[10/15 04:16:34][INFO] (L, W, H) = (312, 812, 720)
/content/py310/lib/python3.10/site-packages/hydra/core/utils.py:212: UserWarning: register_new_resolver() is deprecated and will be removed in a future release.
Use register_resolver() instead.
See https://github.com/hydra-ecosystem/omegaconf/issues/426 for migration instructions.

  OmegaConf.register_new_resolver(
/content/py310/lib/python3.10/site-packages/hydra/core/utils.py:218: UserWarning: register_new_resolver() is deprecated and will be removed in a future release.
Use register_resolver() instead.
See https://github.com/hydra-ecosystem/omegaconf/issues/426 for migration instructions.

  OmegaConf.register_new_resolver(
/content/py310/lib/python3.10/site-packages/hydra/core/utils.py:230: UserWarning: register_new_resolver() is deprecated and will be removed in a future release.
Use register_resolver() instead.
See https://github.com/hydra-ecosystem/ome

In [9]:
# Visualize the result.
display_video('outputs/demo/tennis/tennis_3_incam_global_horiz.mp4')

### Custom Demo

In order to run your own video, you need to follow instructions below:

0. Run the code block below to initialize `demo_google_drive_video()`.
1. Upload your video on Google drive and set accessbility as "Anyone with link (Viewer)".
2. Copy the link and extract the ID. The link should follow this pattern: `https://drive.google.com/file/d/<Google Drive ID>/view?usp=sharing`.
3. Call `demo_google_drive_video()` with the URL, and pass `True` if the camera is static.


In [15]:
%cd {proj_root}
!mkdir -p inputs/demo

def demo_google_drive_video(url:str, static_camera:bool):
    ''' URL should be like https://drive.google.com/file/d/xxxxxxxx/view?usp=drive_link '''

    print(f'[1/3] 📥 Downloading video...')
    gdid = url.split('/')[5]
    video_name = f'custom_{gdid}'
    download_url = f'\'https://drive.google.com/uc?id={gdid}&export=download&confirm=t\''
    !gdown {download_url} -O inputs/demo/{video_name}.mp4

    print(f'[2/3] 💃 Start running GVHMR...')
    flag = '-s' if static_camera else ''

    #point to py310
    !PYTHONPATH= MPLBACKEND=Agg {py310} {proj_root}/tools/demo/demo.py --video=inputs/demo/{video_name}.mp4 {flag}

    print(f'[3/3] 📺 Displaying result...')
    return display_video(f'outputs/demo/{video_name}/{video_name}_3_incam_global_horiz.mp4')

/content/GVHMR


In [16]:
demo_google_drive_video(url='https://drive.google.com/file/d/1KkTsoAuj9yq5JCJ-qg4_MnNBn56wR47F/view?usp=drive_link', static_camera=True)

[1/3] 📥 Downloading video...
Downloading...
From: https://drive.google.com/uc?id=1KkTsoAuj9yq5JCJ-qg4_MnNBn56wR47F&export=download&confirm=t
To: /content/GVHMR/inputs/demo/custom_1KkTsoAuj9yq5JCJ-qg4_MnNBn56wR47F.mp4
100% 936k/936k [00:00<00:00, 107MB/s]
[2/3] 💃 Start running GVHMR...
[10/15 04:25:44][INFO] [Input]: inputs/demo/custom_1KkTsoAuj9yq5JCJ-qg4_MnNBn56wR47F.mp4
[10/15 04:25:44][INFO] (L, W, H) = (367, 682, 666)
/content/py310/lib/python3.10/site-packages/hydra/core/utils.py:212: UserWarning: register_new_resolver() is deprecated and will be removed in a future release.
Use register_resolver() instead.
See https://github.com/hydra-ecosystem/omegaconf/issues/426 for migration instructions.

  OmegaConf.register_new_resolver(
/content/py310/lib/python3.10/site-packages/hydra/core/utils.py:218: UserWarning: register_new_resolver() is deprecated and will be removed in a future release.
Use register_resolver() instead.
See https://github.com/hydra-ecosystem/omegaconf/issues/426 fo

In [19]:
%cd {proj_root}
import re

def demo_uploaded_video(static_camera:bool, path:str=None):
  if path is None:
    from google.colab import files
    print(f'[1/3] 📤 Choose your video from your video...')
    src = list(files.upload().keys())[0]
  else:
    print(f'[1/3] 📤 Using {path}')
    src = path
  video_name = 'upload_' + re.sub(r'[^A-Za-z0-9]+', '_', Path(src).stem).strip('_')
  !mkdir -p inputs/demo && cp "{src}" inputs/demo/{video_name}.mp4 && rm -rf outputs/demo/{video_name}

  print(f'[2/3] 💃 Start running GVHMR...')
  flag = '-s' if static_camera else ''
  !PYTHONPATH= MPLBACKEND=Agg {py310} {proj_root}/tools/demo/demo.py --video=inputs/demo/{video_name}.mp4 {flag}

  print(f'[3/3] 📺 Displaying result...')
  return display_video(f'outputs/demo/{video_name}/{video_name}_3_incam_global_horiz.mp4')



/content/GVHMR


In [22]:
# Try it yourself!
demo_uploaded_video(static_camera=True)

[1/3] 📤 Choose your video from your video...


Saving HWN.MOV to HWN.MOV
[2/3] 💃 Start running GVHMR...
[10/15 04:44:47][INFO] [Input]: inputs/demo/upload_HWN.mp4
[10/15 04:44:47][INFO] (L, W, H) = (501, 1920, 1080)
/content/py310/lib/python3.10/site-packages/hydra/core/utils.py:212: UserWarning: register_new_resolver() is deprecated and will be removed in a future release.
Use register_resolver() instead.
See https://github.com/hydra-ecosystem/omegaconf/issues/426 for migration instructions.

  OmegaConf.register_new_resolver(
/content/py310/lib/python3.10/site-packages/hydra/core/utils.py:218: UserWarning: register_new_resolver() is deprecated and will be removed in a future release.
Use register_resolver() instead.
See https://github.com/hydra-ecosystem/omegaconf/issues/426 for migration instructions.

  OmegaConf.register_new_resolver(
/content/py310/lib/python3.10/site-packages/hydra/core/utils.py:230: UserWarning: register_new_resolver() is deprecated and will be removed in a future release.
Use register_resolver() instead.
S